# Assignment 9 — Natural Language Processing (NLP)

**Sentences used throughout the assignment:**

1. *"The children are playing happily in the gardens."*
2. *"I am running and walking every morning."*
3. *"The organization organized an event."*

## Library and data setup

In [1]:
# If NLTK is missing, run once:  !pip install nltk
import re
import string
from datetime import datetime
import pandas as pd
import nltk

for pkg in ["punkt", "punkt_tab", "wordnet", "stopwords", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet, stopwords

sentences = [
    "The children are playing happily in the gardens.",
    "I am running and walking every morning.",
    "The organization organized an event.",
]
paragraph = " ".join(sentences)
print(paragraph)

The children are playing happily in the gardens. I am running and walking every morning. The organization organized an event.


## Question 1: Sentence and word tokenization

**Sentence tokenization** divides a passage into individual sentences. **Word tokenization** breaks each sentence into separate tokens, which may include words as well as punctuation marks.

In [2]:
# Split the complete paragraph into individual sentences
sent_tokens = sent_tokenize(paragraph)
print("Number of sentences:", len(sent_tokens))
for i, s in enumerate(sent_tokens, 1):
    print(f"  {i}. {s}")

Number of sentences: 3
  1. The children are playing happily in the gardens.
  2. I am running and walking every morning.
  3. The organization organized an event.


In [3]:
# Break each sentence into its word and punctuation tokens
word_tokens = {}
for s in sentences:
    word_tokens[s] = word_tokenize(s)
    print(s)
    print("  Tokens :", word_tokens[s])
    print("  Count  :", len(word_tokens[s]), "(the full stop counts as a token)")
    print()

The children are playing happily in the gardens.
  Tokens : ['The', 'children', 'are', 'playing', 'happily', 'in', 'the', 'gardens', '.']
  Count  : 9 (the full stop counts as a token)

I am running and walking every morning.
  Tokens : ['I', 'am', 'running', 'and', 'walking', 'every', 'morning', '.']
  Count  : 8 (the full stop counts as a token)

The organization organized an event.
  Tokens : ['The', 'organization', 'organized', 'an', 'event', '.']
  Count  : 6 (the full stop counts as a token)



## Question 2: Word segmentation compared with tokenization

| | **Word segmentation** | **Tokenization** |
|---|---|---|
| **Meaning** | Identifying word boundaries when they are not explicitly marked in the text | Dividing text into individual **tokens**, such as words, numbers, punctuation and symbols |
| **Main challenge** | Determining *where* one word finishes and the next begins | Determining what should be treated as a token (for example, whether `don't` is one token or two, and whether `.` is a token) |
| **Common use cases** | Chinese, Japanese and Thai, which may be written without spaces, as well as joined text such as hashtags or URLs: `#naturallanguageprocessing` | Languages such as English and Urdu, where spaces and punctuation usually indicate boundaries |
| **Difficulty** | More challenging because a dictionary or statistical model may be needed, and more than one segmentation can sometimes be possible | Generally simpler because spaces and punctuation provide useful boundary clues |
| **Example** | `ilovenaturallanguageprocessing` → `i love natural language processing` | `"I love NLP."` → `["I", "love", "NLP", "."]` |

Word segmentation can be viewed as a more specialized and demanding form of tokenization. In English, spaces already indicate most word boundaries, so tokenization mainly deals with punctuation and special cases. Segmentation becomes especially important when those spaces are absent.

In [4]:
# Example: dictionary-based word segmentation using maximum matching
dictionary = {"i", "love", "natural", "language", "processing", "nlp", "the", "children", "play", "in", "garden"}

def segment(text, vocab, max_len=12):
    words, i = [], 0
    while i < len(text):
        for j in range(min(len(text), i + max_len), i, -1):     # try the longest match first
            if text[i:j] in vocab:
                words.append(text[i:j])
                i = j
                break
        else:                                                   # unknown character
            words.append(text[i])
            i += 1
    return words

joined = "ilovenaturallanguageprocessing"
print("Joined text        :", joined)
print("Word segmentation  :", segment(joined, dictionary))
print()
print("Normal text        : I love natural language processing.")
print("Tokenization       :", word_tokenize("I love natural language processing."))

Joined text        : ilovenaturallanguageprocessing
Word segmentation  : ['i', 'love', 'natural', 'language', 'processing']

Normal text        : I love natural language processing.
Tokenization       : ['I', 'love', 'natural', 'language', 'processing', '.']


## Question 3: Stemming

**Stemming** reduces words by removing their endings (suffixes) according to a set of fixed rules. The resulting *stem* does not necessarily have to be a valid dictionary word. In this section, the Porter and Snowball stemmers are used to process the given sentences.

In [5]:
porter = PorterStemmer()
snowball = SnowballStemmer("english")

for s in sentences:
    toks = [t for t in word_tokenize(s) if t not in string.punctuation]
    print(s)
    print("  Porter  :", [porter.stem(t) for t in toks])
    print("  Snowball:", [snowball.stem(t) for t in toks])
    print()

The children are playing happily in the gardens.
  Porter  : ['the', 'children', 'are', 'play', 'happili', 'in', 'the', 'garden']
  Snowball: ['the', 'children', 'are', 'play', 'happili', 'in', 'the', 'garden']

I am running and walking every morning.
  Porter  : ['i', 'am', 'run', 'and', 'walk', 'everi', 'morn']
  Snowball: ['i', 'am', 'run', 'and', 'walk', 'everi', 'morn']

The organization organized an event.
  Porter  : ['the', 'organ', 'organ', 'an', 'event']
  Snowball: ['the', 'organ', 'organ', 'an', 'event']



In [6]:
stem_rows = []
for s in sentences:
    for t in word_tokenize(s):
        if t not in string.punctuation:
            stem_rows.append({"Word": t, "Porter stem": porter.stem(t), "Snowball stem": snowball.stem(t)})
stem_table = pd.DataFrame(stem_rows)
stem_table

,Word,Porter stem,Snowball stem
0,The,the,the
1,children,children,children
2,are,are,are
3,playing,play,play
4,happily,happili,happili
5,in,in,in
6,the,the,the
7,gardens,garden,garden
8,I,i,i
9,am,am,am


## Question 4: Text normalization

**Text normalization** transforms text into a consistent format, allowing different forms of a word to be handled uniformly. The following operations are performed:

1. Change all text to lower case.
2. Split the text into tokens.
3. Remove punctuation marks.
4. **Lemmatize** each token: use its part of speech to convert it to its dictionary form (lemma). For example, the verb *are* becomes *be*, while the noun *children* becomes *child*.

In [7]:
lemmatizer = WordNetLemmatizer()

def to_wordnet_pos(tag):
    if tag.startswith("J"): return wordnet.ADJ
    if tag.startswith("V"): return wordnet.VERB
    if tag.startswith("R"): return wordnet.ADV
    return wordnet.NOUN

def normalize(sentence):
    tokens = word_tokenize(sentence.lower())                         # 1 + 2
    tokens = [t for t in tokens if t not in string.punctuation]      # 3
    tagged = pos_tag(tokens)
    return [lemmatizer.lemmatize(w, to_wordnet_pos(tag)) for w, tag in tagged]   # 4

for s in sentences:
    print("Original  :", s)
    print("Normalized:", normalize(s))
    print("As text   :", " ".join(normalize(s)))
    print()

Original  : The children are playing happily in the gardens.


Normalized: ['the', 'child', 'be', 'play', 'happily', 'in', 'the', 'garden']
As text   : the child be play happily in the garden

Original  : I am running and walking every morning.
Normalized: ['i', 'be', 'run', 'and', 'walk', 'every', 'morning']
As text   : i be run and walk every morning

Original  : The organization organized an event.
Normalized: ['the', 'organization', 'organize', 'an', 'event']
As text   : the organization organize an event



## Question 5: Using regular expressions to identify emails, phone numbers and dates

None of the three original sentences includes an email address, phone number or date. Therefore, a separate sample passage is used to test the regular expressions. It contains examples that should be detected, along with invalid examples that should be rejected.

In [8]:
sample_text = """
Contact us at ali.khan@example.com or support_team@my-company.co.uk for help.
Wrong emails: user@@mail..com, @nouser.com, plain.text, name@domain

Phone numbers: +92 300 1234567, 0300-1234567, 03001234567, +1 (555) 123-4567, 555-123-4567.
Not a phone number: order id 123456789012, year 2024.

Dates: 25/12/2024, 05-03-2023, 2024-12-25, December 25, 2024, 25 Dec 2024, 1st January 2025.
Not a date: 99/99/9999 and the version 1.2.3.
"""
print(sample_text)


Contact us at ali.khan@example.com or support_team@my-company.co.uk for help.
Wrong emails: user@@mail..com, @nouser.com, plain.text, name@domain

Phone numbers: +92 300 1234567, 0300-1234567, 03001234567, +1 (555) 123-4567, 555-123-4567.
Not a phone number: order id 123456789012, year 2024.

Dates: 25/12/2024, 05-03-2023, 2024-12-25, December 25, 2024, 25 Dec 2024, 1st January 2025.
Not a date: 99/99/9999 and the version 1.2.3.



### 5a. Email addresses

The following expression is used to find email addresses:

```
\b[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}\b
```

| Part | Purpose |
|---|---|
| `[A-Za-z0-9._%+-]+` | Matches the local part, containing one or more letters, digits or the symbols `. _ % + -` |
| `@` | Matches the required `@` character |
| `[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*` | Matches the domain name and any optional sub-domains (for example, `my-company`) |
| `\.[A-Za-z]{2,}` | Matches the final dot followed by a top-level domain containing at least two letters, such as `com`, `co.uk` or `org` |

In [9]:
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}\b"
emails = re.findall(email_pattern, sample_text)
print("Emails found:", emails)

Emails found: ['ali.khan@example.com', 'support_team@my-company.co.uk']


### 5b. Phone numbers

This regular expression is designed to recognize several common phone-number layouts:

```
(?<!\d)(?:\+\d{1,3}[\s-]?)?(?:\(\d{3}\)\s?|\d{3,4}[\s-]?)\d{3}[\s-]?\d{4}(?!\d)
```

| Part | Purpose |
|---|---|
| `(?<!\d)` and `(?!\d)` | Ensure that the match is not immediately attached to other digits |
| `(?:\+\d{1,3}[\s-]?)?` | Allows an optional international dialing code, such as `+92` or `+1` |
| `(?:\(\d{3}\)\s?|\d{3,4}[\s-]?)` | Matches an area or operator code, such as `(555)`, `0300` or `555`, with an optional separator |
| `\d{3}[\s-]?\d{4}` | Matches the final seven digits, allowing an optional space or hyphen between the groups |

In [10]:
phone_pattern = r"(?<!\d)(?:\+\d{1,3}[\s-]?)?(?:\(\d{3}\)\s?|\d{3,4}[\s-]?)\d{3}[\s-]?\d{4}(?!\d)"
phones = [m.group().strip() for m in re.finditer(phone_pattern, sample_text)]
print("Phone numbers found:")
for p in phones:
    print("  ", p)

Phone numbers found:
   +92 300 1234567
   0300-1234567
   03001234567
   +1 (555) 123-4567
   555-123-4567


### 5c. Dates

The expression below covers four frequently used date layouts: `dd/mm/yyyy` or `dd-mm-yyyy`, `yyyy-mm-dd`, `Month dd, yyyy`, and `dd Month yyyy`. It restricts the day to the range 01–31 and the numeric month to 01–12.

In [11]:
months = r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
day    = r"(?:0?[1-9]|[12]\d|3[01])"
month  = r"(?:0?[1-9]|1[0-2])"

date_pattern = (
    rf"\b{day}[/-]{month}[/-]\d{{4}}\b"                          # 25/12/2024, 05-03-2023
    rf"|\b\d{{4}}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])\b"   # 2024-12-25
    rf"|\b{months}\.? {day}(?:st|nd|rd|th)?,? \d{{4}}\b"        # December 25, 2024
    rf"|\b{day}(?:st|nd|rd|th)? {months}\.?,? \d{{4}}\b"        # 25 Dec 2024, 1st January 2025
)
dates = [m.group() for m in re.finditer(date_pattern, sample_text)]
print("Dates found:")
for d in dates:
    print("  ", d)

Dates found:
   25/12/2024
   05-03-2023
   2024-12-25
   December 25, 2024
   25 Dec 2024
   1st January 2025


In [12]:
# Regex verifies the date's layout only; datetime can check whether it exists on the calendar:
for d in ["25/12/2024", "31/02/2024"]:
    try:
        datetime.strptime(d, "%d/%m/%Y")
        print(f"{d}: valid date")
    except ValueError:
        print(f"{d}: matches the pattern but is NOT a real date (February has no 31st)")

25/12/2024: valid date
31/02/2024: matches the pattern but is NOT a real date (February has no 31st)


In [13]:
# Display a compact summary of the detected items
pd.DataFrame({
    "Type": ["Emails", "Phone numbers", "Dates"],
    "Found": [len(emails), len(phones), len(dates)],
    "Matches": [emails, phones, dates],
})

,Type,Found,Matches
0,Emails,2,"[ali.khan@example.com, support_team@my-company..."
1,Phone numbers,5,"[+92 300 1234567, 0300-1234567, 03001234567, +..."
2,Dates,6,"[25/12/2024, 05-03-2023, 2024-12-25, December ..."


## Question 6: Evaluating stemming and normalization results

In [14]:
rows = []
for s in sentences:
    toks = [t for t in word_tokenize(s) if t not in string.punctuation]
    lemmas = normalize(s)
    for t, l in zip(toks, lemmas):
        rows.append({"Original": t, "Stem (Porter)": porter.stem(t), "Normalized (lower + lemma)": l})
compare = pd.DataFrame(rows)
compare

,Original,Stem (Porter),Normalized (lower + lemma)
0,The,the,the
1,children,children,child
2,are,are,be
3,playing,play,play
4,happily,happili,happily
5,in,in,in
6,the,the,the
7,gardens,garden,garden
8,I,i,i
9,am,am,be


In [15]:
all_words = [w.lower() for w in compare["Original"]]
print("Unique words      :", len(set(all_words)))
print("Unique stems      :", len(set(compare["Stem (Porter)"])))
print("Unique lemmas     :", len(set(compare["Normalized (lower + lemma)"])))
print()

stop_words = set(stopwords.words("english"))
not_words = [s for s in set(compare["Stem (Porter)"]) if s not in stop_words and not wordnet.synsets(s)]
print("Stems that are not dictionary words (stopwords ignored):", sorted(not_words))
print()
print("Words reduced to the same stem:")
print(compare.groupby("Stem (Porter)")["Original"].apply(list)[lambda x: x.str.len() > 1].to_string())

Unique words      : 18
Unique stems      : 17
Unique lemmas     : 17

Stems that are not dictionary words (stopwords ignored): ['everi', 'happili']

Words reduced to the same stem:
Stem (Porter)
organ    [organization, organized]
the                [The, the, The]


## Overall comparison: stemming and normalization

| Word | Stemming (Porter) | Normalization (lower case + lemma) | More suitable result |
|---|---|---|---|
| children | children (unchanged) | **child** | Normalization: recognizes an irregular plural form |
| are, am | are, am (unchanged) | **be**, **be** | Normalization: maps both irregular verb forms to the same lemma |
| happily | happili (not a word) | happily | Normalization: retains a valid dictionary word |
| every | everi (not a word) | every | Normalization |
| morning | morn | morning | Normalization: preserves the original meaning |
| organization | organ | organization | Normalization: *organ* has a different meaning |
| organized | organ | organize | Normalization |
| playing, running, walking, gardens | play, run, walk, garden | play, run, walk, garden | Both methods produce the same result |

| Aspect | Stemming | Normalization (including lemmatization) |
|---|---|---|
| **Approach** | Removes word endings using fixed rules | Uses a dictionary together with each word's part of speech |
| **Result** | Produces stems that may **not** be real words, such as `happili` and `everi` | Produces dictionary forms, or lemmas, that are generally valid words |
| **Processing speed** | Usually faster and simpler | Usually slower because it requires a dictionary and part-of-speech tagging |
| **Irregular forms** | Does not reliably handle forms such as `children`, `are` and `am` | Can handle them, producing `child` and `be` |
| **Possible errors** | **Over-stemming:** `organization` and `organized` both become `organ`, changing the intended meaning. **Under-stemming:** `children` is not reduced to `child` | Can still make mistakes if a word receives an incorrect part-of-speech tag |
| **Additional processing** | Primarily shortens words | Also applies lower-casing and punctuation removal to make the text more consistent |

**Conclusion:** Both approaches reduce variation among word forms. In this example, the 18 unique words are reduced to 17 unique stems and 17 unique lemmas, but the methods achieve this differently. Stemming combines *organization* and *organized* into `organ`, which does not preserve their intended meaning, and it does not connect *are* with *am*. Normalization, on the other hand, maps *are* and *am* to *be*, changes *children* to *child*, and keeps the resulting words readable.

**Stemming** is useful when processing speed is important and exact word forms are less important, such as in some search-engine applications. **Normalization with lemmatization** is preferable when preserving meaning matters, including tasks such as chatbots, text classification and translation.